# NB01 - Data Collection

Project question: How have the most popular movies changed across the decades (1960-2020) in terms of genres, runtime, and rating?

Main data source: [The Movie Database (TMDB) API](https://developer.themoviedb.org/) - a public authenticated API (requires a free key).

What this notebook does

This notebook *only collects and saves* the raw data. Specifically, it:

1. Downloads the TMDB genre table (id -> name).
2. For each year in the chosen range, requests the most-voted movies (`/discover/movie` endpoint).
3. Enriches each movie with its details (`/movie/{id}`) to obtain the runtime, budget, revenue, language, and countries.

All responses are saved as-is (JSON) under `data/raw/tmdb/`, so that NB02 can start from them without calling the API again.



### Why this question and this source

I chose this question because runtime, genre, and audience rating are three simple, comparable signals of how mainstream cinema evolves, and there is a common perception that films have grown longer and more concentrated in a few blockbuster genres over time, a claim worth testing against data rather than intuition. TMDB is well suited to answering it because each film record exposes exactly the fields the question needs: a release_date (to place the film in a decade), a list of genres, the runtime in minutes, and an aggregated vote_average together with a vote_count. Its /discover/movie endpoint lets me sample films systematically, year by year, ranked by popularity, so I can assemble a consistent cross-decade dataset from a single, well-documented API.

## 1. Setup and imports

Load the libraries, read the key from `.env`, and prepare the output folders.


In [1]:
import os
import json
import time
from pathlib import Path

import requests
from dotenv import load_dotenv

# Load the variables defined in the .env file (in the project root)
load_dotenv()

API_KEY = os.getenv("API_KEY")

BASE_URL = "https://api.themoviedb.org/3"

# Output folders for the raw data.
# The notebook lives in notebooks/, so we go up one level to reach data/.
RAW_DIR = Path("..") / "data" / "raw" / "tmdb"
DISCOVER_DIR = RAW_DIR / "discover"   # one response per (year, page)
DETAILS_DIR = RAW_DIR / "details"     # one response per movie

for d in (RAW_DIR, DISCOVER_DIR, DETAILS_DIR):
    d.mkdir(parents=True, exist_ok=True)




## 2. Collection parameters

Everything you might want to change is here at the top.

- START_YEAR / END_YEAR: time range to study.
- PAGES_PER_YEAR: each TMDB page returns 20 movies.
- MIN_VOTE_COUNT: discard movies with very few votes (noise).


### Why these parameters

I study 1960-2020 because it spans six full decades of sound-era popular cinema while staying within the period TMDB documents reliably; earlier years hold far fewer catalogued films with votes. I require at least 100 votes (MIN_VOTE_COUNT) so that each film's rating rests on a meaningful number of viewers rather than a handful, which keeps vote_average stable and comparable across eras. Taking only one page per year (PAGES_PER_YEAR = 1, the ~20 most-voted films of each year) is a deliberate trade-off: I gain a clean, comparable sample of each year's best-known films and a fast, reproducible pipeline, but I lose the long tail of smaller releases, so my findings describe popular cinema rather than total output.


In [2]:
START_YEAR = 1960
END_YEAR = 2020
PAGES_PER_YEAR = 1        # 20 movies per year; raise to 2-3 to widen the sample
MIN_VOTE_COUNT = 100      # ignore movies with fewer votes than this

LANGUAGE = "en-US"        # language of the metadata (titles, genre names)
PAUSE = 0.25              # seconds to wait between requests (courtesy to the API)

print(f"Collecting movies from {START_YEAR} to {END_YEAR}, "
      f"{PAGES_PER_YEAR} page(s) per year, with >= {MIN_VOTE_COUNT} votes.")


## 3. Helper function to call the API

Instead of repeating the same `requests.get(...)` block over and over, we define a single function that:

- automatically adds the key and the language,
- waits a little between calls so as not to overload the API,
- retries if TMDB responds with "too many requests" (code 429),
- raises a clear error if something goes wrong.


In [3]:
def tmdb_get(endpoint, params=None):
    """Call a TMDB endpoint and return the response as a dictionary.

    params:   additional query parameters (dict).
    """
    url = BASE_URL + endpoint
    query = {"api_key": API_KEY, "language": LANGUAGE}
    if params:
        query.update(params)

    for attempt in range(5):
        resp = requests.get(url, params=query, timeout=30)

        # 429 = too many requests: wait as long as the header asks and retry
        if resp.status_code == 429:
            wait = int(resp.headers.get("Retry-After", 2))
            print(f"  Rate limit reached; waiting {wait}s and retrying...")
            time.sleep(wait + 1)
            continue

        resp.raise_for_status()   # raise an error if the code is 4xx/5xx
        time.sleep(PAUSE)
        return resp.json()

    


## 4. Genre table

TMDB stores genres as ids on each movie. We download the id -> name table once and save it for NB02.

In [4]:
genres = tmdb_get("/genre/movie/list")

with open(RAW_DIR / "genres.json", "w", encoding="utf-8") as f:
    json.dump(genres, f, ensure_ascii=False, indent=2)

print(f"Saved {len(genres.get('genres', []))} genres to {RAW_DIR / 'genres.json'}")
genres


Saved 19 genres to ../data/raw/tmdb/genres.json


{'genres': [{'id': 28, 'name': 'Action'},
  {'id': 12, 'name': 'Adventure'},
  {'id': 16, 'name': 'Animation'},
  {'id': 35, 'name': 'Comedy'},
  {'id': 80, 'name': 'Crime'},
  {'id': 99, 'name': 'Documentary'},
  {'id': 18, 'name': 'Drama'},
  {'id': 10751, 'name': 'Family'},
  {'id': 14, 'name': 'Fantasy'},
  {'id': 36, 'name': 'History'},
  {'id': 27, 'name': 'Horror'},
  {'id': 10402, 'name': 'Music'},
  {'id': 9648, 'name': 'Mystery'},
  {'id': 10749, 'name': 'Romance'},
  {'id': 878, 'name': 'Science Fiction'},
  {'id': 10770, 'name': 'TV Movie'},
  {'id': 53, 'name': 'Thriller'},
  {'id': 10752, 'name': 'War'},
  {'id': 37, 'name': 'Western'}]}

## 5. Most-voted movies per year (`/discover/movie`)

For each year we request movies ordered by number of votes (most to least), keeping the most established ones. We save **each page as-is** in `data/raw/tmdb/discover/`.

Along the way, we accumulate the `id` of every movie for the next step (the details).


In [5]:
movie_ids = set()

for year in range(START_YEAR, END_YEAR + 1):
    for page in range(1, PAGES_PER_YEAR + 1):
        target = DISCOVER_DIR / f"{year}_p{page}.json"

        # Resumable: if we already downloaded it in a previous run, reuse it
        if target.exists():
            data = json.loads(target.read_text(encoding="utf-8"))
        else:
            data = tmdb_get("/discover/movie", {
                "sort_by": "vote_count.desc",
                "primary_release_year": year,
                "vote_count.gte": MIN_VOTE_COUNT,
                "page": page,
            })
            with open(target, "w", encoding="utf-8") as f:
                json.dump(data, f, ensure_ascii=False, indent=2)

        for movie in data.get("results", []):
            movie_ids.add(movie["id"])

    print(f"{year}: collected. Unique movies so far: {len(movie_ids)}")

print(f"\nDone. {len(movie_ids)} unique movies to download details for.")


1960: collected. Unique movies so far: 20
1961: collected. Unique movies so far: 40
1962: collected. Unique movies so far: 60
1963: collected. Unique movies so far: 80
1964: collected. Unique movies so far: 100
1965: collected. Unique movies so far: 120
1966: collected. Unique movies so far: 140
1967: collected. Unique movies so far: 160
1968: collected. Unique movies so far: 180
1969: collected. Unique movies so far: 200
1970: collected. Unique movies so far: 220
1971: collected. Unique movies so far: 240
1972: collected. Unique movies so far: 260
1973: collected. Unique movies so far: 280
1974: collected. Unique movies so far: 300
1975: collected. Unique movies so far: 320
1976: collected. Unique movies so far: 340
1977: collected. Unique movies so far: 360
1978: collected. Unique movies so far: 380
1979: collected. Unique movies so far: 400
1980: collected. Unique movies so far: 420
1981: collected. Unique movies so far: 440
1982: collected. Unique movies so far: 460
1983: collected

1985: collected. Unique movies so far: 520
1986: collected. Unique movies so far: 540
1987: collected. Unique movies so far: 560
1988: collected. Unique movies so far: 580
1989: collected. Unique movies so far: 600
1990: collected. Unique movies so far: 620
1991: collected. Unique movies so far: 640
1992: collected. Unique movies so far: 660
1993: collected. Unique movies so far: 680
1994: collected. Unique movies so far: 700
1995: collected. Unique movies so far: 720
1996: collected. Unique movies so far: 740
1997: collected. Unique movies so far: 760
1998: collected. Unique movies so far: 780
1999: collected. Unique movies so far: 800
2000: collected. Unique movies so far: 820
2001: collected. Unique movies so far: 840
2002: collected. Unique movies so far: 860
2003: collected. Unique movies so far: 880
2004: collected. Unique movies so far: 900
2005: collected. Unique movies so far: 920
2006: collected. Unique movies so far: 940
2007: collected. Unique movies so far: 960
2008: colle

## 6. Details for each movie (`/movie/{id}`)

The previous listing does not include the runtime or financial data. We request them movie by movie and save them in `data/raw/tmdb/details/`.

This is the longest step (one request per movie). Because it is resumable, if it gets interrupted you can re-run the cell and it will continue where it left off.


In [6]:
ids = sorted(movie_ids)
total = len(ids)
new = 0

for i, movie_id in enumerate(ids, start=1):
    target = DETAILS_DIR / f"{movie_id}.json"
    if target.exists():
        continue   # already downloaded

    data = tmdb_get(f"/movie/{movie_id}")
    with open(target, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    new += 1

    if i % 50 == 0 or i == total:
        print(f"  {i}/{total} processed ({new} new downloads)")

print(f"\nDetails complete: {new} new movies saved to {DETAILS_DIR}")



Details complete: 0 new movies saved to ../data/raw/tmdb/details


## 7. Quick sanity check

A minimal look to confirm the data is on disk and has the fields we expect. 

### What the collection produced

The run saved raw responses for 1,220 unique films across 1960-2020: 61 discover pages (one per year) and 1,220 detail files. Because I request a single page per year, the sample is deliberately held at roughly 20 films per year, which keeps it comparable across decades even though TMDB's underlying catalogue is far larger for recent years than for the 1960s. Inspecting one record (for example GoldenEye, 1995) confirms that the fields the analysis needs are present: runtime, genres, vote_average, and vote_count. A few records store runtime, budget, or revunue as 0, TMDB's placeholder for "not recorded", which I convert to missing values in NB02 rather than here. The rate limit was not reached during the run.


In [7]:
n_discover = len(list(DISCOVER_DIR.glob("*.json")))
n_details = len(list(DETAILS_DIR.glob("*.json")))
print(f"Discover pages saved: {n_discover}")
print(f"Movie detail files saved: {n_details}")

# Look at one example record and its key fields
example = next(DETAILS_DIR.glob("*.json"))
record = json.loads(example.read_text(encoding="utf-8"))
print(f"\nExample: {record.get('title')} ({record.get('release_date')})")
print("Available fields (sample):",
      [k for k in ("title","release_date","runtime","genres","vote_average",
                   "vote_count","budget","revenue","original_language") if k in record])


Discover pages saved: 61
Movie detail files saved: 1220

Example: GoldenEye (1995-11-16)
Available fields (sample): ['title', 'release_date', 'runtime', 'genres', 'vote_average', 'vote_count', 'budget', 'revenue', 'original_language']


## Summary

This notebook has saved, under `data/raw/tmdb/`:

- `genres.json` - translation of genre id to name.
- `discover/` - one response per year/page with the most-voted movies.
- `details/` - one full record per movie (includes runtime, rating, budget, etc.).